# Phase 3: Gemini AI Classification

Classify Iran-related Trump posts with 5-dimension scoring using the Gemini API with Google Search grounding for fact-checking. Each post is scored on escalation intent, de-escalation intent, fabrication risk, market mover probability, and timing suspicion.

## Step 1: Load Iran Posts

Load cleaned Iran posts and split into crisis-period (Feb-Mar 2026) and older comparison samples. Filter out very short posts and ReTruths.

In [8]:
import os
import json
import time
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()

PROCESSED = '../data/processed'

In [9]:
iran_posts = pd.read_csv(os.path.join(PROCESSED, 'iran_posts_cleaned.csv'))
iran_posts['timestamp'] = pd.to_datetime(iran_posts['timestamp'])

# Focus on the critical period: Feb-Mar 2026 (when the crisis was active)
# Plus sample older posts for comparison
crisis_posts = iran_posts[iran_posts['timestamp'] >= '2026-02-01'].copy()
older_posts = iran_posts[
    (iran_posts['timestamp'] >= '2025-10-01') &
    (iran_posts['timestamp'] < '2026-02-01')
].sample(min(50, len(iran_posts[iran_posts['timestamp'] < '2026-02-01'])), random_state=42)

posts_to_classify = pd.concat([older_posts, crisis_posts]).sort_values('timestamp').reset_index(drop=True)

# Remove very short posts and ReTruths (less useful for classification)
posts_to_classify = posts_to_classify[
    (posts_to_classify['post_text'].str.len() > 20) &
    (posts_to_classify['is_retruth'] == False)
].reset_index(drop=True)

print(f"Posts to classify: {len(posts_to_classify)}")
print(f"  Crisis period (Feb-Mar 2026): {len(crisis_posts)}")
print(f"  Older sample: {len(older_posts)}")

Posts to classify: 359
  Crisis period (Feb-Mar 2026): 309
  Older sample: 50


## Step 2: Set Up Gemini API

Initialize the Google Gemini client with API key from `.env`. Uses **Gemini 3.1 Flash Lite Preview** with Google Search grounding for fact-checking claims against Iran.

In [10]:
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')
if not GEMINI_API_KEY:
    raise ValueError("No GEMINI_API_KEY in .env")

from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL = 'gemini-3.1-flash-lite-preview'

# Test the API with Google Search grounding
try:
    test = client.models.generate_content(
        model=MODEL,
        contents=[types.Content(
            role="user",
            parts=[types.Part.from_text(text='Say "API working" in exactly 2 words.')]
        )],
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="MINIMAL"),
            tools=[types.Tool(googleSearch=types.GoogleSearch())],
        ),
    )
    print(f"API test: {test.text.strip()}")
except Exception as e:
    raise RuntimeError(f"Gemini API test failed: {e}")

print(f"Using model: {MODEL}")

API test: API working
Using model: gemini-3.1-flash-lite-preview


## Step 3: Define Classification Prompt

The prompt instructs Gemini to score each post across five forensic dimensions (0-100 each), with Google Search grounding used to verify factual claims.

In [11]:
CLASSIFICATION_PROMPT = """You are a forensic analyst. Classify the following Truth Social post from Donald Trump.

POST: "{post_text}"
TIMESTAMP: {timestamp}

Score each dimension 0-100:

1. ESCALATION_INTENT: Does this post threaten military action, sanctions, or economic harm against Iran?
2. DE_ESCALATION_INTENT: Does this post suggest peace talks, deals, or reduced tensions?
3. FABRICATION_RISK: Could this claim be independently verified? Is there evidence this is false? Use Google Search to verify factual claims. Check if Iran officials have confirmed/denied talks.
4. MARKET_MOVER_PROBABILITY: Would a reasonable oil trader expect this to move crude oil prices?
5. TIMING_SUSPICION: Is this post timed for market impact? (weekend, pre-market, after-hours, during trading)

OUTPUT FORMAT (JSON only, no other text):
{{
    "escalation_intent": <score 0-100>,
    "escalation_reasoning": "<1-2 sentences>",
    "de_escalation_intent": <score 0-100>,
    "de_escalation_reasoning": "<1-2 sentences>",
    "fabrication_risk": <score 0-100>,
    "fabrication_reasoning": "<1-2 sentences>",
    "market_mover_probability": <score 0-100>,
    "market_mover_reasoning": "<1-2 sentences>",
    "timing_suspicion": <score 0-100>,
    "timing_suspicion_reasoning": "<1-2 sentences>"
}}"""

## Step 4: Classify Posts via Gemini API (Parallel)

Uses a thread pool with 30 concurrent workers to classify posts in parallel.
Gemini 3.1 Flash Lite allows 359 RPM / 2,740 RPD — more than enough for ~274 posts.
Checkpoints every 50 posts so we can resume if interrupted.

In [12]:
import concurrent.futures

MAX_WORKERS = 100

def classify_one_post(post_text, timestamp):
    """Send a single post to Gemini for classification. Returns parsed dict or None."""
    prompt = CLASSIFICATION_PROMPT.format(
        post_text=str(post_text)[:1000],
        timestamp=str(timestamp)
    )
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=[types.Content(
                role="user",
                parts=[types.Part.from_text(text=prompt)]
            )],
            config=types.GenerateContentConfig(
                thinking_config=types.ThinkingConfig(thinking_level="MINIMAL"),
                tools=[types.Tool(googleSearch=types.GoogleSearch())],
            ),
        )
        text = response.text.strip()
        # Clean markdown code blocks if present
        if text.startswith('```'):
            text = text.split('\n', 1)[1]
            if text.endswith('```'):
                text = text[:-3]
            text = text.strip()
        return json.loads(text)
    except Exception as e:
        return {'error': str(e)[:200]}

def build_result(row, scores):
    """Build a result dict from a row and parsed scores."""
    if 'error' in scores and len(scores) == 1:
        return {
            'post_id': str(row['post_id']),
            'timestamp': str(row['timestamp']),
            'post_text': str(row['post_text'])[:500],
            'escalation_intent': 0, 'de_escalation_intent': 0,
            'fabrication_risk': 0, 'market_mover_probability': 0,
            'timing_suspicion': 0,
            'error': scores['error']
        }
    return {
        'post_id': str(row['post_id']),
        'timestamp': str(row['timestamp']),
        'post_text': str(row['post_text'])[:500],
        'escalation_intent': scores.get('escalation_intent', 0),
        'escalation_reasoning': scores.get('escalation_reasoning', ''),
        'de_escalation_intent': scores.get('de_escalation_intent', 0),
        'de_escalation_reasoning': scores.get('de_escalation_reasoning', ''),
        'fabrication_risk': scores.get('fabrication_risk', 0),
        'fabrication_reasoning': scores.get('fabrication_reasoning', ''),
        'market_mover_probability': scores.get('market_mover_probability', 0),
        'market_mover_reasoning': scores.get('market_mover_reasoning', ''),
        'timing_suspicion': scores.get('timing_suspicion', 0),
        'timing_suspicion_reasoning': scores.get('timing_suspicion_reasoning', ''),
    }

def _classify_worker(args):
    """Worker function for thread pool."""
    idx, row = args
    scores = classify_one_post(row['post_text'], row['timestamp'])
    return idx, build_result(row, scores)

# Check for partial results
partial_path = os.path.join(PROCESSED, 'gemini_partial.csv')
results = {}
start_idx = 0

if os.path.exists(partial_path):
    existing = pd.read_csv(partial_path)
    for i, rec in existing.iterrows():
        results[i] = rec.to_dict()
    start_idx = len(results)
    print(f"Resuming from checkpoint at index {start_idx}")

# Build work items (skip already-classified)
work = [(i, posts_to_classify.iloc[i]) for i in range(start_idx, len(posts_to_classify))]
total = len(posts_to_classify)

print(f"Classifying {len(work)} posts with {MAX_WORKERS} concurrent workers...")
print(f"Estimated time: ~{len(work) / MAX_WORKERS * 1.5 / 60:.1f} minutes\n")

errors = 0
checkpoint_every = 50

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_classify_worker, w): w[0] for w in work}
    done_count = start_idx

    for future in concurrent.futures.as_completed(futures):
        try:
            idx, result = future.result()
            results[idx] = result
            if 'error' in result:
                errors += 1
        except Exception as e:
            errors += 1
            print(f"  Worker exception: {e}")

        done_count += 1
        if done_count % checkpoint_every == 0:
            sorted_results = [results[k] for k in sorted(results.keys())]
            pd.DataFrame(sorted_results).to_csv(partial_path, index=False)
            print(f"  Checkpoint: {done_count}/{total} classified ({errors} errors)")

print(f"\nClassification complete: {done_count}/{total} posts, {errors} errors")

Resuming from checkpoint at index 40
Classifying 319 posts with 100 concurrent workers...
Estimated time: ~0.1 minutes

  Checkpoint: 50/359 classified (0 errors)
  Checkpoint: 100/359 classified (0 errors)
  Checkpoint: 150/359 classified (0 errors)
  Checkpoint: 200/359 classified (0 errors)
  Checkpoint: 250/359 classified (0 errors)
  Checkpoint: 300/359 classified (0 errors)
  Checkpoint: 350/359 classified (0 errors)

Classification complete: 359/359 posts, 0 errors


## Step 5: Save Classifications and Review

Merge partial and new results, deduplicate, and save the final classifications. Print summary statistics for each scoring dimension and the top fabrication-risk posts.

In [13]:
all_results = pd.concat([existing, pd.DataFrame(results)], ignore_index=True) if not existing.empty else pd.DataFrame(results)
all_results = all_results.drop_duplicates(subset='post_id', keep='last')

save_path = os.path.join(PROCESSED, 'gemini_classifications.csv')
all_results.to_csv(save_path, index=False)
print(f"Saved {len(all_results)} classifications to {save_path}")
print(f"Errors: {errors}")

# Stats
if len(all_results) > 0:
    score_cols = ['escalation_intent', 'de_escalation_intent', 'fabrication_risk',
                  'market_mover_probability', 'timing_suspicion']
    for col in score_cols:
        if col in all_results.columns:
            vals = pd.to_numeric(all_results[col], errors='coerce')
            print(f"  {col}: mean={vals.mean():.1f}, max={vals.max():.0f}")

    # Top fabrication risk posts
    all_results['fabrication_risk'] = pd.to_numeric(all_results['fabrication_risk'], errors='coerce')
    top_fab = all_results.nlargest(5, 'fabrication_risk')
    print(f"\nTop 5 fabrication risk posts:")
    for _, r in top_fab.iterrows():
        print(f"  [{r['timestamp'][:10]}] Score: {r['fabrication_risk']:.0f} | {str(r['post_text'])[:60]}...")

Saved 41 classifications to ../data/processed/gemini_classifications.csv
Errors: 0
  escalation_intent: mean=12.0, max=95
  de_escalation_intent: mean=9.5, max=100
  fabrication_risk: mean=24.5, max=95
  market_mover_probability: mean=25.0, max=95
  timing_suspicion: mean=34.0, max=90

Top 5 fabrication risk posts:
  [2025-12-12] Score: 95 | I had a very good conversation this morning with the Prime M...
  [2025-12-30] Score: 95 | The polls are rigged even more than the writers. The real nu...
  [2025-10-01] Score: 90 | As I determined on September 27th, when I activated and call...
  [2025-11-29] Score: 90 | Tariffs have made our Country Rich, Strong, Powerful, and Sa...
  [2025-11-03] Score: 85 | America First Patriot Michael Whatley will be an amazing Uni...
